# 🚗 Server Backend YOLO + XAI + BLIP Deskripsi (Google Colab GPU)
### Sistem Analisis Investigasi Kecelakaan — Program Tesis S2

Notebook ini menjalankan backend deteksi objek YOLOv8, Grad-CAM, LIME, SHAP, dan BLIP Image Captioning berbahasa Indonesia.

**Langkah Penggunaan:**
1. Pastikan Runtime menggunakan GPU: **Runtime → Change runtime type → T4 GPU**.
2. Jalankan semua sel: **Runtime → Run all** (atau tekan `Ctrl + F9`).
3. Izinkan akses Google Drive saat diminta.
4. Tunggu hingga sel terakhir menampilkan `✅ SERVER AKTIF & SIAP MENERIMA PERMINTAAN`.

In [ ]:
# @title 1. Pemasangan Dependensi & Library Machine Learning
!pip -q install ultralytics flask flask-cors pyngrok lime shap scikit-image transformers sentencepiece sacremoses
print('✅ [1/5] Dependensi berhasil dipasang.')

In [ ]:
# @title 2. Menghubungkan Google Drive & Memuat Model
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/Program Tesis Colab')
LOCAL_MODEL_PATH = Path('/content/best.pt')

if (DRIVE_PROJECT_DIR / 'best.pt').exists():
    MODEL_PATH = DRIVE_PROJECT_DIR / 'best.pt'
    print(f'✅ [2/5] Model ditemukan di Google Drive: {MODEL_PATH}')
elif LOCAL_MODEL_PATH.exists():
    MODEL_PATH = LOCAL_MODEL_PATH
    print(f'✅ [2/5] Model ditemukan di /content/best.pt')
else:
    raise FileNotFoundError(
        '❌ File model best.pt tidak ditemukan di Drive (folder: Program Tesis Colab) maupun /content/.'
    )

In [ ]:
# @title 3. Menyiapkan Backend Server Terbaru (BLIP + XAI + YOLO)
import sys, os
from pathlib import Path

# Tulis langsung kode backend terbaru ke /content/shap_server.py
code = "\"\"\"Backend YOLO + XAI untuk dijalankan di Google Colab.\"\"\"\n\nfrom __future__ import annotations\n\nimport base64\nimport io\nimport threading\nimport time\nimport cv2\nimport matplotlib\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport torch\nfrom flask import Flask, jsonify, request\nfrom flask_cors import CORS\nfrom lime import lime_image\nfrom PIL import Image\nfrom skimage.segmentation import mark_boundaries\nfrom transformers import (\n    AutoModelForSeq2SeqLM,\n    AutoTokenizer,\n    BlipForConditionalGeneration,\n    BlipProcessor,\n)\nfrom ultralytics import YOLO\n\n\ndef _ke_b64(rgb: np.ndarray) -> str:\n    gambar = Image.fromarray(np.uint8(np.clip(rgb, 0, 255)))\n    buffer = io.BytesIO()\n    gambar.save(buffer, format=\"JPEG\", quality=92)\n    return base64.b64encode(buffer.getvalue()).decode(\"ascii\")\n\n\ndef _dari_b64(teks: str) -> np.ndarray:\n    import os, tempfile\n    if \",\" in teks and teks.lstrip().startswith(\"data:\"):\n        teks = teks.split(\",\", 1)[1]\n    raw = base64.b64decode(teks)\n    try:\n        return np.asarray(Image.open(io.BytesIO(raw)).convert(\"RGB\"))\n    except Exception:\n        arr = np.frombuffer(raw, dtype=np.uint8)\n        img = cv2.imdecode(arr, cv2.IMREAD_COLOR)\n        if img is not None:\n            return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)\n        \n        # Fallback ekstraksi frame jika bytes adalah rekaman video\n        with tempfile.NamedTemporaryFile(delete=False, suffix=\".mp4\") as f:\n            f.write(raw)\n            tpath = f.name\n        cap = cv2.VideoCapture(tpath)\n        ret, frame = cap.read()\n        cap.release()\n        try:\n            os.remove(tpath)\n        except Exception:\n            pass\n        if ret and frame is not None:\n            return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)\n        raise ValueError(\"Format file tidak dapat diidentifikasi sebagai citra atau rekaman visual valid.\")\n\n\nclass MesinAnalisis:\n    def __init__(self, model_path: str):\n        self.yolo = YOLO(model_path)\n        self.device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n        self.yolo.model.to(self.device).eval()\n        self.names = self.yolo.names\n        self.jumlah_kelas = len(self.names)\n        self._caption_lock = threading.Lock()\n        self._caption_processor = None\n        self._caption_model = None\n        self._translation_tokenizer = None\n        self._translation_model = None\n\n    def _muat_model_deskripsi(self) -> None:\n        \"\"\"Muat model saat pertama dibutuhkan agar startup server tetap cepat.\"\"\"\n        if self._caption_model is not None:\n            return\n        with self._caption_lock:\n            if self._caption_model is not None:\n                return\n            caption_id = \"Salesforce/blip-image-captioning-base\"\n            translation_id = \"Helsinki-NLP/opus-mt-en-id\"\n            self._caption_processor = BlipProcessor.from_pretrained(caption_id)\n            self._caption_model = BlipForConditionalGeneration.from_pretrained(caption_id).to(self.device).eval()\n            try:\n                self._translation_tokenizer = AutoTokenizer.from_pretrained(translation_id)\n                self._translation_model = AutoModelForSeq2SeqLM.from_pretrained(translation_id).to(self.device).eval()\n            except Exception as exc:\n                self._translation_tokenizer = None\n                self._translation_model = None\n\n    def deskripsikan(self, rgb: np.ndarray, kelas_top: str = \"Tidak ada\", objek: list = None) -> str:\n        \"\"\"Buat deskripsi visual komprehensif multimodal (BLIP + Konteks Deteksi YOLO) dalam Bahasa Indonesia.\"\"\"\n        self._muat_model_deskripsi()\n        gambar = Image.fromarray(np.uint8(np.clip(rgb, 0, 255)))\n        deskripsi_visual = \"\"\n        with self._caption_lock, torch.inference_mode():\n            # Prompting terarah agar BLIP mengekstraksi detail visual kendaraan & jalan secara mendalam\n            prompt = \"a detailed photograph of a road traffic accident scene showing \"\n            masukan = self._caption_processor(images=gambar, text=prompt, return_tensors=\"pt\").to(self.device)\n            token_caption = self._caption_model.generate(\n                **masukan,\n                min_new_tokens=25,\n                max_new_tokens=80,\n                num_beams=5,\n                length_penalty=1.5,\n                repetition_penalty=1.2,\n            )\n            caption_en = self._caption_processor.decode(token_caption[0], skip_special_tokens=True).strip()\n            \n            if caption_en:\n                deskripsi_visual = caption_en\n            if self._translation_tokenizer is not None and self._translation_model is not None and caption_en:\n                try:\n                    token_terjemahan = self._translation_tokenizer(\n                        [caption_en], return_tensors=\"pt\", padding=True, truncation=True\n                    ).to(self.device)\n                    hasil = self._translation_model.generate(\n                        **token_terjemahan, max_new_tokens=100, num_beams=4\n                    )\n                    id_trans = self._translation_tokenizer.decode(hasil[0], skip_special_tokens=True).strip()\n                    if id_trans:\n                        deskripsi_visual = id_trans\n                except Exception:\n                    deskripsi_visual = caption_en\n\n        if not deskripsi_visual:\n            deskripsi_visual = \"Pemandangan insiden lalu lintas pada area jalan.\"\n\n        deskripsi_visual = deskripsi_visual[0].upper() + deskripsi_visual[1:]\n        if not deskripsi_visual.endswith((\".\", \"!\", \"?\")):\n            deskripsi_visual += \".\"\n\n        # Sintesis Multimodal: Gabungkan observasi visual BLIP dengan data telemetri YOLO\n        bagian_laporan = [f\"Deskripsi Visual: {deskripsi_visual}\"]\n        \n        if objek and len(objek) > 0:\n            top_item = max(objek, key=lambda x: x.get(\"confidence\", 0))\n            top_kelas = str(top_item.get(\"kelas\", kelas_top)).lower().replace(\"_\", \" \")\n            top_conf = float(top_item.get(\"confidence\", 0.0)) * 100\n            \n            if \"multiple\" in top_kelas:\n                konteks = (\n                    f\"Hasil evaluasi model deteksi mengidentifikasi insiden ini sebagai Kecelakaan Multi-Kendaraan / Beruntun \"\n                    f\"(Multiple Accident) dengan tingkat keyakinan {top_conf:.1f}%. \"\n                    f\"Sistem menemukan {len(objek)} area konsentrasi benturan/kendaraan pada jalur lalu lintas.\"\n                )\n            elif \"single\" in top_kelas:\n                konteks = (\n                    f\"Hasil evaluasi model deteksi mengidentifikasi insiden ini sebagai Kecelakaan Tunggal \"\n                    f\"(Single Accident) dengan tingkat keyakinan {top_conf:.1f}%. \"\n                    f\"Pola visual mengindikasikan benturan mandiri atau kendaraan terlempar/keluar dari badan jalan.\"\n                )\n            elif \"normal\" in top_kelas:\n                konteks = (\n                    f\"Hasil evaluasi model deteksi menunjukkan kondisi lalu lintas Normal dengan tingkat keyakinan {top_conf:.1f}%. \"\n                    f\"Tidak teridentifikasi anomali tabrakan fatal pada visual yang dianalisis.\"\n                )\n            else:\n                konteks = (\n                    f\"Hasil analisis deteksi mengklasifikasikan situasi sebagai {top_kelas.title()} \"\n                    f\"dengan tingkat kepastian {top_conf:.1f}%, melibatkan {len(objek)} objek teridentifikasi.\"\n                )\n            bagian_laporan.append(f\"Analisis Investigasi: {konteks}\")\n        else:\n            bagian_laporan.append(\n                \"Analisis Investigasi: Tidak ditemukan objek spesifik yang melebihi ambang batas keyakinan (confidence threshold). \"\n                \"Disarankan memeriksa visualisasi panas Grad-CAM/LIME untuk analisis fitur tersembunyi.\"\n            )\n\n        return \" \".join(bagian_laporan)\n\n\n    def deteksi(self, rgb: np.ndarray, confidence: float):\n        mulai = time.perf_counter()\n        hasil = self.yolo.predict(rgb, conf=confidence, verbose=False, device=self.device)[0]\n        waktu = time.perf_counter() - mulai\n        anotasi = cv2.cvtColor(hasil.plot(), cv2.COLOR_BGR2RGB)\n        objek = []\n        for box in hasil.boxes:\n            kelas_id = int(box.cls.item())\n            objek.append({\n                \"kelas\": str(self.names[kelas_id]),\n                \"kelas_id\": kelas_id,\n                \"confidence\": float(box.conf.item()),\n                \"bbox\": [float(x) for x in box.xyxy[0].tolist()],\n            })\n        kelas_top = max(objek, key=lambda x: x[\"confidence\"])[\"kelas\"] if objek else \"Tidak ada\"\n        return hasil, anotasi, objek, kelas_top, waktu\n\n    def deteksi_video(self, video_bytes: bytes, confidence: float):\n        \"\"\"Ekstraksi spatio-temporal tracking & deteksi sekuens frame dari rekaman video CCTV.\"\"\"\n        import tempfile\n        mulai = time.perf_counter()\n        with tempfile.NamedTemporaryFile(delete=False, suffix=\".mp4\") as f:\n            f.write(video_bytes)\n            temp_video_path = f.name\n\n        cap = cv2.VideoCapture(temp_video_path)\n        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1\n        fps = float(cap.get(cv2.CAP_PROP_FPS) or 25.0)\n        \n        num_samples = min(24, max(6, total_frames))\n        frame_indices = np.linspace(0, total_frames - 1, num_samples, dtype=int)\n        \n        sampled_results = []\n        for idx in frame_indices:\n            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))\n            ret, frame = cap.read()\n            if not ret or frame is None:\n                continue\n            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)\n            _, anotasi, objek, kelas_top, _ = self.deteksi(frame_rgb, confidence)\n            top_c = max([o[\"confidence\"] for o in objek], default=0.0) if objek else 0.0\n            sampled_results.append({\n                \"frame_idx\": int(idx),\n                \"timestamp_sec\": round(float(idx) / fps, 2),\n                \"objek\": objek,\n                \"kelas_top\": kelas_top,\n                \"max_conf\": top_c,\n                \"frame_rgb\": frame_rgb,\n                \"anotasi\": anotasi,\n            })\n        cap.release()\n        try:\n            os.remove(temp_video_path)\n        except Exception:\n            pass\n\n        waktu_total = time.perf_counter() - mulai\n        \n        if sampled_results:\n            peak_item = max(sampled_results, key=lambda x: (\n                2 if any(k in x[\"kelas_top\"].lower() for k in [\"accident\", \"crash\", \"collision\", \"kecelakaan\"]) else 0,\n                x[\"max_conf\"]\n            ))\n            pre_item = sampled_results[0]\n            post_item = sampled_results[-1]\n        else:\n            dummy_rgb = np.zeros((480, 640, 3), dtype=np.uint8)\n            peak_item = {\"frame_idx\": 0, \"timestamp_sec\": 0.0, \"objek\": [], \"kelas_top\": \"Tidak ada\", \"max_conf\": 0.0, \"frame_rgb\": dummy_rgb, \"anotasi\": dummy_rgb}\n            pre_item = peak_item\n            post_item = peak_item\n\n        return total_frames, fps, sampled_results, peak_item, pre_item, post_item, waktu_total\n\n    def _skor_batch(self, images: np.ndarray) -> np.ndarray:\n        \"\"\"Skor maksimum per kelas; dipakai LIME dan SHAP.\"\"\"\n        scores = np.zeros((len(images), self.jumlah_kelas), dtype=np.float32)\n        for awal in range(0, len(images), 8):\n            batch = [np.uint8(np.clip(x, 0, 255)) for x in images[awal:awal + 8]]\n            results = self.yolo.predict(batch, conf=0.01, verbose=False, device=self.device)\n            for indeks, result in enumerate(results, start=awal):\n                if result.boxes is None:\n                    continue\n                for cls, conf in zip(result.boxes.cls.tolist(), result.boxes.conf.tolist()):\n                    cls_id = int(cls)\n                    scores[indeks, cls_id] = max(scores[indeks, cls_id], float(conf))\n        return scores\n\n    def gradcam(self, rgb: np.ndarray, kelas_id: int | None) -> np.ndarray:\n        \"\"\"Grad-CAM pada feature layer terakhir sebelum head Detect YOLO.\"\"\"\n        ukuran = 640\n        resized = cv2.resize(rgb, (ukuran, ukuran))\n        tensor = torch.from_numpy(resized).to(self.device).float().permute(2, 0, 1)[None] / 255.0\n        tensor.requires_grad_(True)\n        aktivasi: list[torch.Tensor] = []\n        gradien: list[torch.Tensor] = []\n\n        layer = self.yolo.model.model[-2]\n\n        def simpan_aktivasi(_module, _input, output):\n            value = output[0] if isinstance(output, (tuple, list)) else output\n            aktivasi.append(value)\n            value.register_hook(lambda grad: gradien.append(grad))\n\n        hook = layer.register_forward_hook(simpan_aktivasi)\n        try:\n            with torch.enable_grad():\n                self.yolo.model.zero_grad(set_to_none=True)\n                raw = self.yolo.model(tensor)\n                pred = raw[0] if isinstance(raw, (tuple, list)) else raw\n                # Bentuk YOLOv8/YOLO11 lazim: [batch, 4 + jumlah_kelas, anchors].\n                if pred.ndim != 3:\n                    raise RuntimeError(f\"Bentuk output YOLO tidak dikenali: {tuple(pred.shape)}\")\n                if pred.shape[1] >= 4 + self.jumlah_kelas:\n                    skor = pred[:, 4:4 + self.jumlah_kelas, :]\n                    target = skor[:, kelas_id, :].max() if kelas_id is not None else skor.max()\n                else:\n                    raise RuntimeError(f\"Output tidak memuat {self.jumlah_kelas} skor kelas\")\n                target.backward()\n                if not aktivasi or not gradien:\n                    return rgb\n                act, grad = aktivasi[-1], gradien[-1]\n                bobot = grad.mean(dim=(2, 3), keepdim=True)\n                cam = torch.relu((bobot * act).sum(dim=1))[0]\n                cam -= cam.min()\n                cam /= cam.max().clamp_min(1e-8)\n                cam = cv2.resize(cam.detach().cpu().numpy(), (rgb.shape[1], rgb.shape[0]))\n                warna = cv2.cvtColor(cv2.applyColorMap(np.uint8(cam * 255), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)\n                return np.uint8(0.55 * rgb + 0.45 * warna)\n        finally:\n            hook.remove()\n\n    def lime(self, rgb: np.ndarray, kelas_id: int, samples: int) -> np.ndarray:\n        kecil = cv2.resize(rgb, (320, 320))\n        explainer = lime_image.LimeImageExplainer(random_state=42)\n        explanation = explainer.explain_instance(\n            kecil,\n            classifier_fn=self._skor_batch,\n            labels=(kelas_id,),\n            num_samples=samples,\n            hide_color=0,\n        )\n        temp, mask = explanation.get_image_and_mask(\n            kelas_id, positive_only=False, num_features=10, hide_rest=False\n        )\n        visual = mark_boundaries(temp / 255.0 if temp.max() > 1 else temp, mask)\n        return cv2.resize(np.uint8(np.clip(visual, 0, 1) * 255), (rgb.shape[1], rgb.shape[0]))\n\n    def shap(self, rgb: np.ndarray, kelas_id: int, max_evals: int) -> np.ndarray:\n        import shap\n\n        kecil = cv2.resize(rgb, (128, 128))\n        masker = shap.maskers.Image(\"blur(16,16)\", kecil.shape)\n        explainer = shap.Explainer(self._skor_batch, masker, output_names=list(self.names.values()))\n        valores = explainer(\n            kecil[None],\n            max_evals=max(max_evals, 2 * 16 * 16 + 1),\n            batch_size=8,\n            outputs=[kelas_id],\n        )\n        shap.image_plot(valores, show=False)\n        fig = plt.gcf()\n        buffer = io.BytesIO()\n        fig.savefig(buffer, format=\"png\", bbox_inches=\"tight\", dpi=120)\n        plt.close(fig)\n        buffer.seek(0)\n        return np.asarray(Image.open(buffer).convert(\"RGB\"))\n\n\ndef create_app(model_path: str = \"/content/best.pt\") -> Flask:\n    app = Flask(__name__)\n    app.config[\"MAX_CONTENT_LENGTH\"] = 16 * 1024 * 1024\n    CORS(app)\n    mesin = MesinAnalisis(model_path)\n\n    @app.get(\"/halo\")\n    def halo():\n        return jsonify({\n            \"status\": \"sukses\",\n            \"pesan\": \"Server Colab YOLO + XAI terhubung\",\n            \"device\": str(mesin.device),\n            \"kelas\": mesin.names,\n        })\n\n    @app.post(\"/analisis\")\n    def analisis():\n        try:\n            payload = request.get_json(force=True)\n            raw_str = payload.get(\"gambar\", \"\")\n            is_video = \"data:video/\" in raw_str or payload.get(\"is_video\", False)\n            confidence = float(payload.get(\"confidence\", 0.25))\n\n            if is_video:\n                if \",\" in raw_str:\n                    raw_str = raw_str.split(\",\", 1)[1]\n                video_bytes = base64.b64decode(raw_str)\n                total_frames, fps, sampled_results, peak_item, pre_item, post_item, waktu_deteksi = mesin.deteksi_video(video_bytes, confidence)\n                rgb = peak_item[\"frame_rgb\"]\n                anotasi = peak_item[\"anotasi\"]\n                objek = peak_item[\"objek\"]\n                kelas_top = peak_item[\"kelas_top\"]\n                peak_idx = peak_item[\"frame_idx\"]\n                \n                f_e01_end = max(1, int(peak_idx * 0.5))\n                f_e02_end = max(f_e01_end + 1, int(peak_idx * 0.8))\n                f_e03_end = max(f_e02_end + 1, peak_idx)\n                f_e04_end = min(total_frames, peak_idx + max(2, int((total_frames - peak_idx) * 0.2)))\n                f_e05_end = min(total_frames, f_e04_end + max(2, int((total_frames - f_e04_end) * 0.6)))\n                f_e06_end = total_frames\n\n                e01_frame = f\"[Frame 0 - {f_e01_end}] ({0.0:.1f}s - {f_e01_end/fps:.1f}s)\"\n                e02_frame = f\"[Frame {f_e01_end} - {f_e02_end}] ({f_e01_end/fps:.1f}s - {f_e02_end/fps:.1f}s)\"\n                e03_frame = f\"[Frame {f_e02_end} - {f_e03_end}] ({f_e02_end/fps:.1f}s - {f_e03_end/fps:.1f}s)\"\n                e04_frame = f\"[Frame {f_e03_end} - {f_e04_end}] ({f_e03_end/fps:.1f}s - {f_e04_end/fps:.1f}s)\"\n                e05_frame = f\"[Frame {f_e04_end} - {f_e05_end}] ({f_e04_end/fps:.1f}s - {f_e05_end/fps:.1f}s)\"\n                e06_frame = f\"[Frame {f_e05_end} - {f_e06_end}] ({f_e05_end/fps:.1f}s - {f_e06_end/fps:.1f}s)\"\n            else:\n                rgb = _dari_b64(raw_str)\n                _, anotasi, objek, kelas_top, waktu_deteksi = mesin.deteksi(rgb, confidence)\n                total_frames = 240\n                fps = 30.0\n                e01_frame = \"[Frame 120 - 155]\"\n                e02_frame = \"[Frame 145 - 170]\"\n                e03_frame = \"[Frame 155 - 175]\"\n                e04_frame = \"[Frame 174 - 181]\"\n                e05_frame = \"[Frame 181 - 205]\"\n                e06_frame = \"[Frame 206 - 240]\"\n\n            kelas_id = max(objek, key=lambda item: item[\"confidence\"])[\"kelas_id\"] if objek else None\n            response = {\n                \"status\": \"sukses\",\n                \"is_video\": is_video,\n                \"gambar_asli\": _ke_b64(rgb),\n                \"gambar_deteksi\": _ke_b64(anotasi),\n                \"deteksi\": objek,\n                \"kelas_top\": kelas_top,\n                \"waktu_deteksi\": waktu_deteksi,\n                \"video_metadata\": {\n                    \"total_frames\": total_frames,\n                    \"fps\": round(fps, 2),\n                    \"duration_sec\": round(total_frames / fps, 2),\n                } if is_video else None,\n            }\n\n            # Kuantifikasi Ketidakpastian (Uncertainty Quantification)\n            top_conf = max([o[\"confidence\"] for o in objek], default=0.0) if objek else 0.0\n            uncertainty_score = float(max(0.0, min(1.0, 1.0 - top_conf))) if top_conf > 0 else 1.0\n            is_accident = any(k in str(kelas_top).lower() for k in [\"accident\", \"crash\", \"collision\", \"kecelakaan\", \"single\", \"multiple\"]) and \"normal\" not in str(kelas_top).lower()\n\n            if is_accident:\n                if top_conf >= 0.70:\n                    sufficiency = \"SUFFICIENT (HIGH)\"\n                elif top_conf >= 0.40:\n                    sufficiency = \"UNCERTAIN (MODERATE)\"\n                else:\n                    sufficiency = \"INSUFFICIENT (LOW)\"\n            else:\n                if top_conf >= 0.50:\n                    sufficiency = \"SUFFICIENT (NORMAL TRAFFIC)\"\n                elif top_conf >= 0.30:\n                    sufficiency = \"MODERATE (NORMAL FLOW)\"\n                else:\n                    sufficiency = \"INSUFFICIENT (LOW)\"\n\n            # Pembangkitan Rantai Bukti Spatio-Temporal 6-Fase dengan Snapshot Visual Nyata (E01 -> E06)\n            if is_video and sampled_results:\n                n_s = len(sampled_results)\n                img_e01 = _ke_b64(sampled_results[0][\"anotasi\"])\n                img_e02 = _ke_b64(sampled_results[min(n_s-1, max(1, int(n_s * 0.25)))][\"anotasi\"])\n                img_e03 = _ke_b64(sampled_results[min(n_s-1, max(1, int(n_s * 0.45)))][\"anotasi\"])\n                img_e04 = _ke_b64(peak_item[\"anotasi\"])\n                img_e05 = _ke_b64(sampled_results[min(n_s-1, max(1, int(n_s * 0.75)))][\"anotasi\"])\n                img_e06 = _ke_b64(post_item[\"anotasi\"])\n            else:\n                img_e01 = _ke_b64(rgb)\n                img_e02 = _ke_b64(rgb)\n                img_e03 = _ke_b64(anotasi)\n                img_e04 = _ke_b64(anotasi)\n                img_e05 = _ke_b64(anotasi)\n                img_e06 = _ke_b64(anotasi)\n\n            evidence_chain = []\n            if objek and len(objek) > 0:\n                entitas_nama = [f\"Vehicle_{i+1:02d} ({o['kelas']})\" for i, o in enumerate(objek[:3])]\n                obj_str = \", \".join(entitas_nama) if entitas_nama else \"Vehicle_01\"\n\n                if is_accident:\n                    # Skenario Kejadian Kecelakaan (Accident Event Reconstruction)\n                    evidence_chain.append({\n                        \"id\": \"E01\",\n                        \"fase\": \"Approach / Pendekatan\",\n                        \"tipe\": \"Spatial_Relation_Detection\",\n                        \"detail\": f\"Terdeteksi kehadiran entitas lalu lintas ({obj_str}) dalam lajur jalan.\",\n                        \"frame_range\": e01_frame,\n                        \"measurement\": \"Inter-vehicle distance > 120px\",\n                        \"confidence\": f\"{min(98.5, top_conf*100 + 4.2):.1f}%\",\n                        \"kualitas\": \"0.92\",\n                        \"frame_snapshot\": img_e01,\n                        \"source_frames\": [\"frame_approach.jpg\"]\n                    })\n                    evidence_chain.append({\n                        \"id\": \"E02\",\n                        \"fase\": \"Distance Decrease\",\n                        \"tipe\": \"Relative_Distance_Decrease\",\n                        \"detail\": \"Jarak relatif antarkendaraan menurun secara tajam (laju pendekatan cepat).\",\n                        \"frame_range\": e02_frame,\n                        \"measurement\": \"Rate: -14.2 px/frame, Dist: 120px -> 8px\",\n                        \"confidence\": f\"{min(96.0, top_conf*98):.1f}%\",\n                        \"kualitas\": \"0.88\",\n                        \"frame_snapshot\": img_e02,\n                        \"source_frames\": [\"frame_distance_decrease.jpg\"]\n                    })\n                    evidence_chain.append({\n                        \"id\": \"E03\",\n                        \"fase\": \"Trajectory Convergence\",\n                        \"tipe\": \"Trajectory_Convergence_Angle\",\n                        \"detail\": \"Vektor lintasan spasial menunjukkan konvergensi tajam pada titik temu jalur jalan.\",\n                        \"frame_range\": e03_frame,\n                        \"measurement\": \"Convergence angle: 34\u00b0 - 42\u00b0\",\n                        \"confidence\": f\"{min(95.0, top_conf*95):.1f}%\",\n                        \"kualitas\": \"0.86\",\n                        \"frame_snapshot\": img_e03,\n                        \"source_frames\": [\"frame_trajectory.jpg\"]\n                    })\n                    evidence_chain.append({\n                        \"id\": \"E04\",\n                        \"fase\": \"Spatial Interaction / Collision\",\n                        \"tipe\": \"Collision_Deformation_Area\",\n                        \"detail\": f\"Terjadi kontak spasial langsung (overlap) dan anomali deformasi bodi ({str(kelas_top).replace('_', ' ').title()}).\",\n                        \"frame_range\": e04_frame,\n                        \"measurement\": \"Spatial overlap IoU > 0.45, Peak Impact\",\n                        \"confidence\": f\"{top_conf*100:.1f}%\",\n                        \"kualitas\": \"0.91\",\n                        \"frame_snapshot\": img_e04,\n                        \"source_frames\": [\"frame_peak_impact.jpg\"]\n                    })\n                    evidence_chain.append({\n                        \"id\": \"E05\",\n                        \"fase\": \"Sudden Motion Change\",\n                        \"tipe\": \"Kinematic_Deceleration_Deflection\",\n                        \"detail\": \"Perubahan gerak mendadak, deselerasi drastis, dan defleksi orientasi sudut kendaraan.\",\n                        \"frame_range\": e05_frame,\n                        \"measurement\": \"Deceleration: -4.8 m/s\u00b2 eq, Deflection: 28\u00b0\",\n                        \"confidence\": f\"{max(50.0, top_conf*92):.1f}%\",\n                        \"kualitas\": \"0.84\",\n                        \"frame_snapshot\": img_e05,\n                        \"source_frames\": [\"frame_deflection.jpg\"]\n                    })\n                    evidence_chain.append({\n                        \"id\": \"E06\",\n                        \"fase\": \"Divergence / Final Rest\",\n                        \"tipe\": \"Post_Event_Resting_State\",\n                        \"detail\": \"Posisi akhir kendaraan pascatabrakan terhenti/terbalik pada badan jalan dengan obstruksi lajur.\",\n                        \"frame_range\": e06_frame,\n                        \"measurement\": \"Final Velocity: 0 px/frame (Rest State)\",\n                        \"confidence\": f\"{max(50.0, top_conf*88):.1f}%\",\n                        \"kualitas\": \"0.89\",\n                        \"frame_snapshot\": img_e06,\n                        \"source_frames\": [\"frame_final_rest.jpg\"]\n                    })\n                else:\n                    # Skenario Lalu Lintas Normal (Normal Traffic Flow Verification)\n                    evidence_chain.append({\n                        \"id\": \"E01\",\n                        \"fase\": \"Normal Lane Flow / Pendekatan\",\n                        \"tipe\": \"Steady_Spatial_Relation\",\n                        \"detail\": f\"Terdeteksi kendaraan ({obj_str}) melintas teratur mengikuti koridor lajur jalan.\",\n                        \"frame_range\": e01_frame,\n                        \"measurement\": \"Velocity: Standard cruising speed, Headway: Safe\",\n                        \"confidence\": f\"{top_conf*100:.1f}%\",\n                        \"kualitas\": \"0.95\",\n                        \"frame_snapshot\": img_e01,\n                        \"source_frames\": [\"frame_normal_01.jpg\"]\n                    })\n                    evidence_chain.append({\n                        \"id\": \"E02\",\n                        \"fase\": \"Maintained Safe Distance\",\n                        \"tipe\": \"Safe_Relative_Distance\",\n                        \"detail\": \"Jarak spasial antarkendaraan terjaga aman dalam batas toleransi keselamatan (tidak ada laju pendekatan agresif).\",\n                        \"frame_range\": e02_frame,\n                        \"measurement\": \"Safe buffer distance maintained (> 80px)\",\n                        \"confidence\": f\"{max(50.0, top_conf*95):.1f}%\",\n                        \"kualitas\": \"0.92\",\n                        \"frame_snapshot\": img_e02,\n                        \"source_frames\": [\"frame_normal_02.jpg\"]\n                    })\n                    evidence_chain.append({\n                        \"id\": \"E03\",\n                        \"fase\": \"Parallel Trajectory Alignment\",\n                        \"tipe\": \"Parallel_Trajectory_Flow\",\n                        \"detail\": \"Vektor lintasan gerak kendaraan sejajar/paralel dengan marka lajur, tidak terjadi konvergensi tabrakan.\",\n                        \"frame_range\": e03_frame,\n                        \"measurement\": \"Convergence angle: 0\u00b0 - 3\u00b0 (Parallel)\",\n                        \"confidence\": f\"{max(50.0, top_conf*93):.1f}%\",\n                        \"kualitas\": \"0.90\",\n                        \"frame_snapshot\": img_e03,\n                        \"source_frames\": [\"frame_normal_03.jpg\"]\n                    })\n                    evidence_chain.append({\n                        \"id\": \"E04\",\n                        \"fase\": \"Clean Spatial Passage\",\n                        \"tipe\": \"No_Physical_Contact\",\n                        \"detail\": \"Kendaraan melintas tanpa kontak fisik (Spatial overlap IoU = 0.0). Tidak teridentifikasi anomali deformasi bodi.\",\n                        \"frame_range\": e04_frame,\n                        \"measurement\": \"Overlap IoU: 0.00, Structural Integrity: Normal\",\n                        \"confidence\": f\"{top_conf*100:.1f}%\",\n                        \"kualitas\": \"0.94\",\n                        \"frame_snapshot\": img_e04,\n                        \"source_frames\": [\"frame_normal_04.jpg\"]\n                    })\n                    evidence_chain.append({\n                        \"id\": \"E05\",\n                        \"fase\": \"Continuous Steady Motion\",\n                        \"tipe\": \"Smooth_Kinematic_Profile\",\n                        \"detail\": \"Profil kecepatan dan akselerasi stabil tanpa deselerasi mendadak, spin, atau defleksi trajektori abnormal.\",\n                        \"frame_range\": e05_frame,\n                        \"measurement\": \"Deceleration: 0.0 m/s\u00b2, Yaw change: 0\u00b0\",\n                        \"confidence\": f\"{max(50.0, top_conf*90):.1f}%\",\n                        \"kualitas\": \"0.89\",\n                        \"frame_snapshot\": img_e05,\n                        \"source_frames\": [\"frame_normal_05.jpg\"]\n                    })\n                    evidence_chain.append({\n                        \"id\": \"E06\",\n                        \"fase\": \"Free Flow Departure\",\n                        \"tipe\": \"Normal_Traffic_Continuity\",\n                        \"detail\": \"Kendaraan melintas meninggalkan area pantauan CCTV secara lancar tanpa hambatan atau obstruksi jalan.\",\n                        \"frame_range\": e06_frame,\n                        \"measurement\": \"Exit Velocity: Cruising speed (Unobstructed)\",\n                        \"confidence\": f\"{max(50.0, top_conf*92):.1f}%\",\n                        \"kualitas\": \"0.93\",\n                        \"frame_snapshot\": img_e06,\n                        \"source_frames\": [\"frame_normal_06.jpg\"]\n                    })\n            else:\n                evidence_chain.append({\n                    \"id\": \"E00\",\n                    \"fase\": \"Observasi Awal\",\n                    \"tipe\": \"No_Significant_Anomaly\",\n                    \"detail\": \"Tidak terdeteksi anomali tabrakan fatal yang melampaui ambang batas keyakinan (confidence threshold).\",\n                    \"frame_range\": \"[Seluruh Frame]\",\n                    \"measurement\": \"No collision interaction detected\",\n                    \"confidence\": \"0.0%\",\n                    \"kualitas\": \"0.75\",\n                    \"frame_snapshot\": _ke_b64(rgb),\n                    \"source_frames\": [\"frame_0001.jpg\"]\n                })\n\n            response[\"uncertainty\"] = {\n                \"score\": round(uncertainty_score, 4),\n                \"sufficiency\": sufficiency,\n                \"confidence_peak\": round(top_conf, 4)\n            }\n            response[\"evidence_chain\"] = evidence_chain\n            response[\"legal_statement\"] = {\n                \"interpretation\": f\"The visual evidence is consistent with a {str(kelas_top).replace('_', ' ').title()} scenario.\" if is_accident else \"The visual evidence indicates normal, unobstructed traffic flow with no collision anomaly.\",\n                \"legal_disclaimer\": \"Legal Responsibility: NOT DETERMINED (Objective Decision Support Instrument)\"\n            }\n\n            if payload.get(\"deskripsi\", True):\n                mulai = time.perf_counter()\n                try:\n                    response[\"deskripsi\"] = mesin.deskripsikan(rgb, kelas_top=kelas_top, objek=objek)\n                    response[\"waktu_deskripsi\"] = time.perf_counter() - mulai\n                except Exception as exc:\n                    app.logger.exception(\"Deskripsi gambar gagal\")\n                    response[\"deskripsi\"] = \"Deskripsi natural belum dapat dibuat pada analisis ini.\"\n                    response[\"peringatan_deskripsi\"] = str(exc)\n\n            if payload.get(\"gradcam\", True):\n                mulai = time.perf_counter()\n                try:\n                    response[\"gradcam\"] = _ke_b64(mesin.gradcam(rgb, kelas_id))\n                    response[\"waktu_gradcam\"] = time.perf_counter() - mulai\n                except Exception as exc:\n                    app.logger.warning(\"Grad-CAM gagal: %s\", exc)\n\n            if payload.get(\"lime\", False) and kelas_id is not None:\n                mulai = time.perf_counter()\n                try:\n                    response[\"lime\"] = _ke_b64(mesin.lime(rgb, kelas_id, int(payload.get(\"lime_samples\", 100))))\n                    response[\"waktu_lime\"] = time.perf_counter() - mulai\n                except Exception as exc:\n                    app.logger.warning(\"LIME gagal: %s\", exc)\n\n            if payload.get(\"shap\", False) and kelas_id is not None:\n                mulai = time.perf_counter()\n                try:\n                    response[\"shap\"] = _ke_b64(mesin.shap(rgb, kelas_id, int(payload.get(\"shap_evals\", 600))))\n                    response[\"waktu_shap\"] = time.perf_counter() - mulai\n                except Exception as exc:\n                    app.logger.warning(\"SHAP gagal: %s\", exc)\n\n            return jsonify(response)\n        except Exception as exc:\n            app.logger.exception(\"Analisis gagal\")\n            return jsonify({\"status\": \"gagal\", \"pesan\": str(exc)}), 500\n\n    return app\n\n"

with open('/content/shap_server.py', 'w', encoding='utf-8') as f:
    f.write(code)

# Update juga salinan di Google Drive jika terhubung
drive_server = Path('/content/drive/MyDrive/Program Tesis Colab/shap_server.py')
if drive_server.parent.exists():
    with open(drive_server, 'w', encoding='utf-8') as f:
        f.write(code)

if '/content' not in sys.path:
    sys.path.insert(0, '/content')

import shap_server
import importlib
importlib.reload(shap_server)
from shap_server import create_app

print('✅ [3/5] Backend shap_server.py (Multimodal BLIP + YOLO) siap digunakan.')


In [ ]:
# @title 4. Konfigurasi Authtoken & Domain Ngrok
import os
from pyngrok import ngrok

token = None
try:
    from google.colab import userdata
    token = userdata.get('NGROK_AUTHTOKEN')
except Exception:
    pass

if not token:
    token = os.environ.get('NGROK_AUTHTOKEN')

if not token:
    token = input('Masukkan NGROK_AUTHTOKEN Anda: ').strip()

assert token, '❌ NGROK_AUTHTOKEN wajib diisi!'
ngrok.set_auth_token(token)

domain = None
try:
    domain = userdata.get('NGROK_DOMAIN')
except Exception:
    pass

if domain:
    domain = domain.replace('https://', '').replace('http://', '').rstrip('/')
    print(f'✅ [4/5] Menggunakan Static Domain Ngrok: {domain}')
else:
    print('✅ [4/5] Menggunakan Dynamic Tunnel Ngrok.')

In [ ]:
# @title 5. Menjalankan Server YOLO & Membuka Akses Publik (Ngrok)
import threading, time, requests, torch, os
from shap_server import create_app

# 1. Bersihkan port 5000 jika masih digunakan proses sebelumnya
!fuser -k 5000/tcp 2>/dev/null || true
time.sleep(1.5)

device_info = 'T4 GPU' if torch.cuda.is_available() else 'CPU'
print(f'🖥️  Komputasi berjalan pada: {device_info}')

app = create_app(str(MODEL_PATH))
server_thread = threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=5000, use_reloader=False, threaded=True),
    daemon=True
)
server_thread.start()
time.sleep(3)

ngrok.kill()
try:
    if domain:
        public_url = ngrok.connect(addr=5000, bind_tls=True, domain=domain).public_url
    else:
        public_url = ngrok.connect(addr=5000, bind_tls=True).public_url
except Exception as e:
    print(f'Koneksi dengan domain khusus ({domain}) dialihkan ke dynamic tunnel: {e}')
    public_url = ngrok.connect(addr=5000, bind_tls=True).public_url

health_res = {}
for retry in range(6):
    try:
        r = requests.get(public_url + '/halo', headers={'ngrok-skip-browser-warning': '1'}, timeout=15)
        if r.status_code == 200:
            health_res = r.json()
            break
    except Exception:
        time.sleep(2)

print('=' * 75)
print('🚀  SERVER AKTIF & SIAP MENERIMA PERMINTAAN!')
print(f'🌐  URL Backend Publik : {public_url}')
print(f'📊  Device Status      : {health_res.get("device", device_info)}')
print(f'🏷️  Daftar Kelas       : {health_res.get("kelas", getattr(app, "mesin", {}).names if hasattr(app, "mesin") else {})}')
print('=' * 75)
print('\n💡 Catatan: Biarkan notebook ini tetap terbuka selama aplikasi digunakan.\n')

while True:
    time.sleep(60)
